# 01a. Tiền xử lý dữ liệu văn bản (Text Preprocessing)

Pipeline làm sạch, chuẩn hóa và tách từ tiếng Việt cho bộ dữ liệu ViMMSD trước khi đưa vào mô hình ngôn ngữ (PhoBERT).

In [ ]:
# === Bootstrap môi trường (Kaggle / Colab / Local) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "main"
CODE_DIR = "/tmp/IE403"

def get_github_token():
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

print("Working directory:", os.getcwd())

In [ ]:
import json, re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import emoji

from vimmsd.data.preprocessing import clean_text, TextPreprocessor, TEENCODE
from vimmsd.utils.config import load_config

cfg = load_config("configs/base.yaml")
train_json_path = Path(cfg.paths.data_dir) / "vimmsd-train.json"

with open(train_json_path, encoding="utf-8") as f:
    raw_train = json.load(f)

df_train = pd.DataFrame(list(raw_train.values()))
print(f"Dataset size: {len(df_train)} samples loaded from {train_json_path}")
df_train.head()

## 1. Khảo sát phân bố và đặc trưng văn bản

In [ ]:
# Phân bố nhãn và thống kê độ dài
print("=== Label Distribution ===")
print(df_train["label"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

df_train["word_count"] = df_train["caption"].apply(lambda c: len(str(c).split()))
df_train["char_count"] = df_train["caption"].apply(lambda c: len(str(c)))

print("\n=== Caption Length Statistics ===")
print(df_train[["word_count", "char_count"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
# Tần suất các loại nhiễu mạng xã hội
captions = df_train["caption"].fillna("").tolist()

url_cnt = sum(1 for c in captions if re.search(r"https?://|www\.", c))
mention_cnt = sum(1 for c in captions if re.search(r"@\w+", c))
hash_cnt = sum(1 for c in captions if "#" in c)
html_cnt = sum(1 for c in captions if re.search(r"&[a-zA-Z]+;|&#\d+;", c))
repeat_cnt = sum(1 for c in captions if re.search(r"([a-zA-ZÀ-ỹ])\1{2,}", c, re.IGNORECASE))
emoji_cnt = sum(1 for c in captions if emoji.emoji_count(c) > 0)

noise_df = pd.DataFrame([
    {"Feature": "Hashtag (#)", "Count": hash_cnt, "Ratio (%)": hash_cnt / len(captions) * 100},
    {"Feature": "Emoji", "Count": emoji_cnt, "Ratio (%)": emoji_cnt / len(captions) * 100},
    {"Feature": "Repeated Characters (>=3)", "Count": repeat_cnt, "Ratio (%)": repeat_cnt / len(captions) * 100},
    {"Feature": "HTML Entities (&amp;, &#39;)", "Count": html_cnt, "Ratio (%)": html_cnt / len(captions) * 100},
    {"Feature": "URLs", "Count": url_cnt, "Ratio (%)": url_cnt / len(captions) * 100},
    {"Feature": "Mentions (@username)", "Count": mention_cnt, "Ratio (%)": mention_cnt / len(captions) * 100},
])
noise_df.round(2)

In [ ]:
# Top 15 emoji phổ biến nhất
all_emojis = []
for c in captions:
    all_emojis.extend(emoji.distinct_emoji_list(c))
top_emojis = Counter(all_emojis).most_common(15)

emoji_df = pd.DataFrame([
    {"Emoji": em, "Demojize": emoji.demojize(em), "Count": cnt}
    for em, cnt in top_emojis
])
emoji_df

## 2. Kiểm tra trực quan pipeline tiền xử lý (`clean_text`)

In [ ]:
sample_indices = [0, 5, 12, 25, 42, 88, 105, 150, 220, 310]
sample_rows = []

for idx in sample_indices:
    raw = captions[idx]
    cleaned_demojize = clean_text(raw, emoji_mode="demojize", word_segment_=True)
    cleaned_keep = clean_text(raw, emoji_mode="keep", word_segment_=True)
    sample_rows.append({
        "Index": idx,
        "Raw Caption": raw,
        "Cleaned (Demojize + PhoBERT)": cleaned_demojize,
        "Cleaned (Keep Emoji)": cleaned_keep,
    })

comparison_df = pd.DataFrame(sample_rows)
pd.set_option("display.max_colwidth", None)
comparison_df

## 3. Kiểm thử các câu trường hợp biên (Edge Cases)

In [ ]:
test_cases = [
    "Giá 1000đ mà quáaaaa đắt, ko mua đc 😂 https://x.com #review @shop_fake",
    "Phục vụ đỉnh chóp lun á, đợi có 2 tiếng thui à khum quạo tí nào 🐧🐧",
    "Ad ơi cmt này đỉnh vch, nhìn mlem ghê á &amp; chất lượng thật sự",
    "Nhìn cái thời tiết này thik ghê á, mưa ngập tới đầu gối rùi 🥲",
    "Đồ ăn ngonnnn đỉnh cao, ăn xong vô bệnh viện lun ak 🤡",
]

for i, tc in enumerate(test_cases, 1):
    print(f"[Test {i}]")
    print("  Raw:", tc)
    print("  Cleaned:", clean_text(tc, emoji_mode="demojize", word_segment_=True))
    print()

## 4. Kiểm thử cơ chế Cache của `TextPreprocessor`

In [ ]:
import time

preprocessor = TextPreprocessor.from_config(cfg.data.text)
test_texts = captions[:500]

# Xử lý lần 1: Tính toán và lưu cache ra đĩa
t0 = time.time()
res1 = preprocessor.process_all(test_texts, cache_dir="data/processed")
t1 = time.time()
print(f"First run (computing and caching 500 samples): {t1 - t0:.3f}s")

# Xử lý lần 2: Đọc trực tiếp từ cache JSON
t2 = time.time()
res2 = preprocessor.process_all(test_texts, cache_dir="data/processed")
t3 = time.time()
print(f"Second run (loaded from cache): {t3 - t2:.4f}s")
assert res1 == res2, "Kết quả cache không khớp!"
print("Cache verification passed successfully.")

## 5. Kết luận

- **Chuẩn hóa văn bản:** Đã làm sạch mã HTML entities, URL, mention, hashtag và chuẩn hóa các từ teencode tiếng Việt phổ biến mà không gây xung đột ngữ cảnh.
- **Tách từ cho PhoBERT:** Sử dụng `underthesea` nối từ ghép `_` chuẩn xác theo yêu cầu mô hình.
- **Độ dài token:** 95% câu có độ dài dưới 230 từ. Cấu hình `max_length = 256` (hoặc `128`) phù hợp để huấn luyện trên GPU.